In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
fraud_data = pd.read_csv("fraud_detection.csv")

print("Dataset loaded successfully!")
print(fraud_data.head())
print("\nDataset shape:", fraud_data.shape)

Dataset loaded successfully!
   TransactionAmount  TransactionFrequency  International  AccountAge  Fraud
0                100                     2              0          60      0
1               2500                    15              1           5      1
2                 50                     1              0         120      0
3               5000                    20              1           3      1
4                120                     3              0          80      0

Dataset shape: (15, 5)


In [3]:
X = fraud_data.drop("Fraud", axis=1)
y = fraud_data["Fraud"]

print("Features:", X.columns.tolist())
print("Target:", y.name)

Features: ['TransactionAmount', 'TransactionFrequency', 'International', 'AccountAge']
Target: Fraud


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 10
Testing samples: 5


In [5]:
# Decision Tree
dt_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)
dt_model.fit(X_train, y_train)


# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    random_state=42
)
rf_model.fit(X_train, y_train)


# XGBoost
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_train, y_train)

print("All three models trained successfully!")

All three models trained successfully!


In [6]:
dt_pred = dt_model.predict(X_test)
rf_pred = rf_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

dt_prob = dt_model.predict_proba(X_test)[:, 1]
rf_prob = rf_model.predict_proba(X_test)[:, 1]
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

print("Predictions generated successfully!")

Predictions generated successfully!


In [7]:
models = {
    "Decision Tree": (dt_pred, dt_prob),
    "Random Forest": (rf_pred, rf_prob),
    "XGBoost": (xgb_pred, xgb_prob)
}

results = []

for name, (pred, prob) in models.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-Score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

comparison_df = pd.DataFrame(results)

print(comparison_df)

           Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
0  Decision Tree       1.0        1.0     1.0       1.0      1.0
1  Random Forest       1.0        1.0     1.0       1.0      1.0
2        XGBoost       1.0        1.0     1.0       1.0      1.0


In [8]:
comparison_df.to_csv(
    "week3_model_comparison.csv",
    index=False
)

print("Final model comparison saved successfully!")

Final model comparison saved successfully!


In [9]:
best_model = comparison_df.sort_values(
    by=["F1-Score", "ROC-AUC"],
    ascending=False
).iloc[0]

print("🏆 Best Model:", best_model["Model"])
print("\nPerformance:")
print(best_model)

🏆 Best Model: Decision Tree

Performance:
Model        Decision Tree
Accuracy               1.0
Precision              1.0
Recall                 1.0
F1-Score               1.0
ROC-AUC                1.0
Name: 0, dtype: object


## Model Strengths and Weaknesses

### Decision Tree
**Strengths**
- Easy to interpret
- Simple decision rules
- Fast training

**Weaknesses**
- Can overfit
- Less robust than ensemble methods

### Random Forest
**Strengths**
- Combines multiple Decision Trees
- More robust than a single tree
- Reduces the effect of individual tree errors

**Weaknesses**
- Less interpretable than one Decision Tree
- Requires more computational resources

### XGBoost
**Strengths**
- Powerful predictive performance
- Sequentially improves previous errors
- Supports regularization

**Weaknesses**
- More complex to tune
- Less interpretable
- Can overfit with poor parameter choices

## Best Model Analysis

The best model was selected using F1-Score as the primary metric and ROC-AUC as a secondary metric.

F1-Score provides a balance between precision and recall, which is important for fraud detection. ROC-AUC provides an additional measure of how effectively the model separates fraudulent and legitimate transactions.

The selected model provides the strongest overall performance among the three models evaluated in this sprint.

However, the dataset used in this project is small and therefore the results should be treated as a learning experiment rather than production-level evidence.

## Week 3 Engineering Reflection

During Week 3, I progressed from basic classification models to advanced ensemble learning techniques.

I built and evaluated Decision Tree, Random Forest, and XGBoost models for a fraud detection problem.

### What I Learned

- Decision Trees provide simple and interpretable decision rules.
- Random Forest improves robustness by combining multiple decision trees.
- XGBoost uses boosting to sequentially improve model predictions.
- Accuracy alone can be misleading, especially for imbalanced classification problems.
- Precision, recall, F1-Score, and ROC-AUC provide a more complete evaluation.
- Model selection should consider both technical performance and real-world business requirements.

### Engineering Insight

A model with the highest accuracy is not automatically the best model. A production ML system should balance predictive performance, interpretability, robustness, scalability, and business impact.

### Limitation

The fraud dataset used in this learning project is small. Therefore, the results demonstrate the modeling workflow but should not be treated as production-level performance.

### Week 3 Outcome

I learned how to move from training individual models